# BMI Calculator

In [ ]:
# !pip install langgraph langchain dotenv

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List

In [ ]:
# define state dictionary for the agent
class AgentState(TypedDict):
    weight_kg: float
    height_m: float
    bmi: float
    label: str

In [ ]:
# calculate BMI helper function
def calculate_bmi_helper(weight_kg: float, height_m: float) -> float:
    """Calculate BMI given weight in kg and height in meters."""
    return weight_kg / (height_m ** 2)

# create the state graph
def calculate_bmi(state: AgentState) -> AgentState:
    """Calculate BMI and update the state."""
    bmi = calculate_bmi_helper(state["weight_kg"], state["height_m"])
    state["bmi"] = bmi
    return state

In [ ]:
# create the state graph
def label_bmi(state: AgentState) -> AgentState:
    """Label BMI category based on the calculated BMI."""
    bmi = state["bmi"]
    if bmi < 18.5:
        state["label"] = "Underweight"
    elif 18.5 <= bmi < 25:
        state["label"] = "Normal weight"
    elif 25 <= bmi < 30:
        state["label"] = "Overweight"
    else:
        state["label"] = "Obese"
    return state

In [ ]:
# define graph for the agent
graph = StateGraph(AgentState)

# add nodes to the graph
graph.add_node('calculate_bmi', calculate_bmi)
graph.add_node("label_bmi", label_bmi)

# add edge to the graph
graph.add_edge(START, 'calculate_bmi')
graph.add_edge('calculate_bmi', 'label_bmi')
graph.add_edge('label_bmi', END)

# compile the graph
workflow = graph.compile()



In [ ]:
# execute the graph
initial_state = AgentState(weight_kg=70, height_m=1.75, bmi=0, label="")
final_state = workflow.invoke(initial_state)
print(final_state)

In [ ]:
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())